# Parametrización de PSO para CVRP (SR-1, Ai & Kachitvichyanukul 2009)

**Fase 1 de 3** (Parametrización → Experimentación → Análisis), mismo patrón que el proyecto QAP.

Se busca el mejor conjunto de hiperparámetros de PSO ($N$, $w_0$, $w_f$, $c_1$, $c_2$, $v_{max}$) usando **Optuna** (TPE, 60 trials — ver justificación abajo), minimizando el GAP% promedio sobre un conjunto de **instancias de tuning separado del de evaluación** (para evitar sesgo por sobreajuste).

**Presupuesto:** 50.000 evaluaciones de la función objetivo por corrida (mismo orden de magnitud que el paper de referencia y que el proyecto QAP), tanto en esta fase de parametrización como en la experimentación final — esto es posible porque el decode SR-1 está acelerado con Numba (~130.000-200.000 evals/s, validado contra una versión de referencia en Python puro).

**Instancias de tuning:** `A-n34-k5` (n=33, chica), `A-n62-k8` (n=61, mediana), `E-n76-k7` (n=75, grande) — ninguna pertenece al conjunto de evaluación final.

**Presupuesto de Optuna (60 trials):** siguiendo la heurística de ~10x el número de hiperparámetros tuneados (6 aquí: $N,w_0,w_f,c_1,c_2,v_{max}$), y consistente con el default real de `Ayudantia 10/1_parametrizacion/*_optuna.py` (60 trials).

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

import numpy as np
import optuna

import cvrp_common as cc

optuna.logging.set_verbosity(optuna.logging.WARNING)
print("optuna", optuna.__version__)

optuna 4.9.0


In [2]:
TUNING_INSTANCES = ["A-n34-k5", "A-n62-k8", "E-n76-k7"]
EVAL_BUDGET = 50_000  # evaluaciones de la funcion objetivo por corrida
TUNING_SEED = 42      # semilla fija durante el tuning: aisla el efecto de los hiperparametros

instances = {name: cc.load_instance(name) for name in TUNING_INSTANCES}
for name, inst in instances.items():
    print(f"{name}: n={inst.n_customers} m={inst.n_vehicles_ref} capacidad={inst.capacity} BKS={inst.bks}")

A-n34-k5: n=33 m=5 capacidad=100.0 BKS=778.0
A-n62-k8: n=61 m=8 capacidad=100.0 BKS=1288.0
E-n76-k7: n=75 m=7 capacidad=220.0 BKS=682.0


## Espacio de búsqueda

| Parámetro | Rango | Referencia |
|---|---|---|
| $N$ (partículas) | {20, 30, 50, 80, 100} | paper usa 50; QAP usó este mismo conjunto discreto |
| $w_0$ | [0.5, 0.95] | paper usa 0.9 |
| $w_f$ | [0.1, 0.5] | paper usa 0.4 |
| $c_1$ | [0.5, 2.5] | rango estándar PSO canónico |
| $c_2$ | [0.5, 2.5] | rango estándar PSO canónico |
| $v_{max}$ (fracción del rango de posición) | [0.1, 1.0] | — |

Dado el presupuesto fijo de 50.000 evaluaciones, el número de iteraciones se deriva de $N$: $T = \lfloor 50000/N \rfloor$ (igual que en QAP), de modo que todas las configuraciones usan el mismo esfuerzo de cómputo sin importar cuántas partículas elijan.

In [3]:
def objective(trial: optuna.Trial) -> float:
    n_particles = trial.suggest_categorical("n_particles", [20, 30, 50, 80, 100])
    w0 = trial.suggest_float("w0", 0.5, 0.95)
    wf = trial.suggest_float("wf", 0.1, 0.5)
    c1 = trial.suggest_float("c1", 0.5, 2.5)
    c2 = trial.suggest_float("c2", 0.5, 2.5)
    v_max_frac = trial.suggest_float("v_max_frac", 0.1, 1.0)

    n_iter = EVAL_BUDGET // n_particles
    params = cc.PSOParams(
        n_particles=n_particles, n_iter=n_iter,
        w0=w0, wf=wf, c1=c1, c2=c2, v_max=v_max_frac,
    )

    gaps = []
    for name, inst in instances.items():
        res = cc.run_pso(inst, inst.n_vehicles_ref, params, seed=TUNING_SEED)
        gaps.append(cc.gap_pct(res.best_cost, inst.bks))
    return float(np.mean(gaps))

In [4]:
N_TRIALS = 60

study = optuna.create_study(
    study_name="pso_cvrp",
    direction="minimize",
    storage="sqlite:///pso_cvrp_optuna.db",
    load_if_exists=True,
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print("Mejor GAP% promedio (tuning):", study.best_value)
print("Mejores parametros:", study.best_params)

  0%|          | 0/60 [00:00<?, ?it/s]

Mejor GAP% promedio (tuning): 8.543552812761812
Mejores parametros: {'n_particles': 100, 'w0': 0.9116439344453389, 'wf': 0.2781836884248188, 'c1': 2.404465247658618, 'c2': 1.1842635297699007, 'v_max_frac': 0.3762271004522205}


In [5]:
best = study.best_params
best_n_iter = EVAL_BUDGET // best["n_particles"]
best_params_obj = cc.PSOParams(
    n_particles=best["n_particles"], n_iter=best_n_iter,
    w0=best["w0"], wf=best["wf"], c1=best["c1"], c2=best["c2"], v_max=best["v_max_frac"],
)

lines = [
    "Mejores parametros PSO (Optuna, 60 trials, presupuesto 50000 evals):",
    f"  N (particulas)    = {best['n_particles']}",
    f"  T (iteraciones)   = {best_n_iter}",
    f"  w0                = {best['w0']:.4f}",
    f"  wf                = {best['wf']:.4f}",
    f"  c1                = {best['c1']:.4f}",
    f"  c2                = {best['c2']:.4f}",
    f"  v_max (fraccion)  = {best['v_max_frac']:.4f}",
    f"  GAP%% promedio (tuning) = {study.best_value:.4f}",
    "",
    "Detalle por instancia de tuning:",
]
for name, inst in instances.items():
    res = cc.run_pso(inst, inst.n_vehicles_ref, best_params_obj, seed=TUNING_SEED)
    gap = cc.gap_pct(res.best_cost, inst.bks)
    lines.append(f"  {name}: costo={res.best_cost:.1f} BKS={inst.bks} GAP%={gap:.4f}")

summary = "\n".join(lines)
print(summary)
Path("resultados_optuna.txt").write_text(summary, encoding="utf-8")

Mejores parametros PSO (Optuna, 60 trials, presupuesto 50000 evals):
  N (particulas)    = 100
  T (iteraciones)   = 500
  w0                = 0.9116
  wf                = 0.2782
  c1                = 2.4045
  c2                = 1.1843
  v_max (fraccion)  = 0.3762
  GAP%% promedio (tuning) = 8.5436

Detalle por instancia de tuning:
  A-n34-k5: costo=804.0 BKS=778.0 GAP%=3.3419
  A-n62-k8: costo=1390.0 BKS=1288.0 GAP%=7.9193
  E-n76-k7: costo=780.0 BKS=682.0 GAP%=14.3695


475